# Lab 02: Convert Notebooks to Scripts & Run Command Jobs

> **Source:** Microsoft Learning - mslearn-mlops
> **Original files:** [github.com/MicrosoftLearning/mslearn-mlops](https://github.com/MicrosoftLearning/mslearn-mlops)
> **License:** MIT
> **Adapted for:** AI-300 Lab Guide

## Learning Objectives

By the end of this lab, you will be able to:

1. **Convert a notebook to a parameterized script** using `argparse`
2. **Submit a command job** via the Azure ML Python SDK `command()` function
3. **Submit a command job** via the Azure CLI with `az ml job create`
4. **Define jobs in YAML** for reproducible, CI/CD-friendly workflows

**Estimated time:** ~15 minutes
**Estimated cost:** ~$0.50 (compute cluster usage)

## Architecture

![Architecture](architecture.png)

**The progression:** Notebook (exploration) --> Script (parameterized with argparse) --> Command Job (submitted via SDK or YAML/CLI) --> CI/CD Pipeline (automated with GitHub Actions)

This lab covers the first three steps. Lab 06 covers the CI/CD pipeline.

In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception:
    credential = InteractiveBrowserCredential()

ml_client = MLClient.from_config(credential=credential)
ws = ml_client.workspaces.get(ml_client.workspace_name)
print(f"Workspace: {ws.name} ({ws.location})")
compute = ml_client.compute.get("cc-ai300")
print(f"Compute cluster: {compute.name} ({compute.size})")
print("\nAll prerequisites met.")

## Step 1: The Training Script

The key to moving from notebooks to production is **parameterization with `argparse`**. Instead of hardcoding values, the script accepts them as command-line arguments:

```python
parser = argparse.ArgumentParser()
parser.add_argument("--training_data", type=str, help="Path to training data")
parser.add_argument("--reg_rate", type=float, default=0.01, help="Regularization rate")
args = parser.parse_args()
```

This makes the script:
- **Reusable** -- different data or hyperparameters without code changes
- **Testable** -- can be run locally or on a cluster
- **CI/CD-ready** -- parameters can be injected by automation pipelines

> **Exam Tip:** The exam tests the progression pattern: **Notebook --> Script (argparse) --> Command Job --> Pipeline --> CI/CD**. Know each step and why it matters for MLOps maturity.

In [ ]:
with open("scripts/train_model_parameters.py") as f:
    print(f.read())

## Step 2: Submit a Command Job via Python SDK

The `command()` function creates a job definition that runs a script on a remote compute target. Key parameters:

| Parameter | Description |
|-----------|-------------|
| `code` | Local folder containing the script(s) to upload |
| `command` | The shell command to execute |
| `inputs` | Dictionary of named inputs (data assets, parameters) |
| `environment` | The Docker image / conda env to run in |
| `compute` | The compute target (cluster name) |

**Input syntax:** Use `${{inputs.param_name}}` in the `command` string to reference values from the `inputs` dictionary. Azure ML substitutes these at runtime.

> **Exam Tip:** The `${{inputs.param_name}}` syntax is used in both SDK `command()` and YAML job definitions. For data inputs, Azure ML mounts/downloads the data and passes the local path to your script.

In [ ]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes

job = command(
    code="./scripts",
    command="python train_model_parameters.py --training_data ${{inputs.training_data}} --reg_rate ${{inputs.reg_rate}}",
    inputs={
        "training_data": Input(type=AssetTypes.URI_FILE, path="azureml:diabetes-data:1"),
        "reg_rate": 0.01,
    },
    environment="azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    compute="cc-ai300",
    experiment_name="diabetes-training",
    display_name="diabetes-train-sdk",
)

returned_job = ml_client.jobs.create_or_update(job)
print(f"Job submitted: {returned_job.name}")
print(f"Monitor at: {returned_job.studio_url}")

## Step 3: Define a Job in YAML

YAML job definitions are the **preferred approach for CI/CD pipelines** because they are:
- **Version-controlled** -- tracked in Git alongside your code
- **Declarative** -- describe *what* to run, not *how* to submit it
- **CLI-friendly** -- submitted with a single `az ml job create` command

The YAML format mirrors the SDK `command()` parameters, using the same `${{inputs.param_name}}` syntax.

> **Exam Tip:** The `$schema` field at the top of the YAML file enables IDE validation and autocompletion. It points to the Azure ML schema for the specific resource type (e.g., `commandJob.schema.json`).

In [ ]:
with open("scripts/job.yml") as f:
    print(f.read())

### Submit via CLI

You can submit the YAML job definition using the Azure CLI:

```bash
az ml job create --file scripts/job.yml --resource-group <rg-name> --workspace-name <ws-name>
```

Or from within a notebook, use the SDK's `load_job()` function to load the YAML and submit it programmatically:

In [ ]:
from azure.ai.ml import load_job

yaml_job = load_job(source="scripts/job.yml")
returned_job = ml_client.jobs.create_or_update(yaml_job)
print(f"YAML job submitted: {returned_job.name}")
print(f"Monitor at: {returned_job.studio_url}")

## Key Takeaways

1. **Notebooks to scripts** -- convert exploratory notebooks into parameterized `.py` scripts using `argparse` for production readiness
2. **argparse enables flexibility** -- scripts accept `--training_data` and `--reg_rate` as arguments, making them reusable across datasets and experiments
3. **SDK `command()` vs CLI `az ml job create`** -- both submit the same job; SDK is better for interactive use, CLI is better for automation
4. **YAML is preferred for CI/CD** -- declarative, version-controlled, and works with `az ml job create --file` in GitHub Actions workflows
5. **`${{inputs.param_name}}` syntax** -- used in both SDK and YAML to reference job inputs; Azure ML resolves data paths and parameter values at runtime